# Bronze Layer —  Metadata Ingestion
**Catalog:** metadata_governance  
**Schema:** bronze  
**Table:** bronze_metadata   
**Layer:** Bronze (raw ingestion — no transformations applied)  
**Source:** AWS S3 — `s3://test-capstone-project-metadata/metadata_realistic_10k.csv`  

## Step 1 — Read Raw CSV from S3

Ingest the raw metadata file from AWS S3 into a Spark DataFrame for further processing.

In [0]:
df_bronze = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("s3://test-capstone-project-metadata/metadata_realistic_10k.csv")

##Step 2 — Add Audit Columns

Add audit metadata columns to track ingestion activity and data lineage.


In [0]:
from pyspark.sql.functions import current_timestamp, current_date, lit

df_bronze = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("s3://test-capstone-project-metadata/metadata_realistic_10k.csv")

df_bronze = df_bronze \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("load_date", current_date()) \
    .withColumn("record_source", lit("AWS_S3"))

## Step 3 — Save to Bronze Delta Table

Store the raw metadata into a Delta Lake table in the Bronze layer for persistent storage.

In [0]:

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .saveAsTable("metadata_governance.bronze.bronze_metadata")

## Step 4 — Validate Row Count

Verify that records were loaded successfully into the Bronze table.

In [0]:
bronze_df = spark.table(
    "metadata_governance.bronze.bronze_metadata"
)

row_count = bronze_df.count()

print(f"Bronze Row Count: {row_count}")

## Step 5 — Validate Schema

Verify that the Bronze table schema matches the source file structure.

In [0]:
bronze_df.printSchema()

## Step 6 — Validate Audit Columns

Verify that audit columns were successfully added during ingestion.

In [0]:
bronze_df = spark.table(
    "metadata_governance.bronze.bronze_metadata")

display(
    bronze_df.select(
        "ingestion_timestamp",
        "load_date",
        "record_source"
    ).limit(10)
)

## Step 7 - Validate Data Completness

Verify data completeness by checking the number of null or empty values in each column of the Bronze dataset.

In [0]:
from pyspark.sql.functions import col, count, when, isnan, isnull

print("Null check for column: ")
null_counts = bronze_df.select(
    [count(when(isnull(c) | (col(c).cast("string") == ""), c)).alias(c) 
     for c in bronze_df.columns]
)

display(null_counts)